In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.pipeline import ChronosPipelinetsfm_lens
from tsfm_lens.utils import (
    apply_custom_style,
    make_ensemble_from_arrow_dir,
    set_seed,
)


In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../figs", "logit_attribution")
os.makedirs(plot_save_dir, exist_ok=True)


In [ ]:
split_name = "base40"
split_dir = os.path.join(DATA_DIR, split_name)

system_name = "Lorenz"
subsample_interval = 1

ensemble = make_ensemble_from_arrow_dir(
    split_dir, dyst_names_lst=[system_name], num_samples=1
)

In [ ]:
context_length = 512
context_start_time = 1280
context_end_time = context_start_time + context_length

In [ ]:
# Prepare input series
sample_idx = 0
selected_dim = 0
# get sample sample_idx of Lorenz
dyst_coords = torch.tensor(ensemble["Lorenz"][sample_idx, selected_dim, :]).unsqueeze(0)
print(dyst_coords.shape)
dyst_coords = dyst_coords[:, ::subsample_interval]
context = dyst_coords[:, context_start_time:context_end_time]

print(context.shape)

In [ ]:
prediction_length = 512

In [ ]:
future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
print(f"future_vals shape: {future_vals.shape}")

### Set up Pipeline

In [ ]:
pipeline = ChronosPipelinetsfm_lens.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.bfloat16,
)
num_layers = pipeline.model.model.config.num_decoder_layers
num_heads = pipeline.model.model.config.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
rseed = 123
set_seed(rseed)

In [ ]:
num_samples = 20

prediction_kwargs = {
    "limit_prediction_length": False,
    "deterministic": True if num_samples == 1 else False,
    "top_k": 50,
    "top_p": 1.0,
    "temperature": 1.0,
    "num_samples": num_samples,
}

print(prediction_kwargs)

In [ ]:
preds, corrected_timesteps = pipeline.predict_with_assimilation(
    context,
    future_vals,
    prediction_length=512,
    **prediction_kwargs,
    latency_delay=5,
    verbose=False,
)

In [ ]:
print(f"Number of interventions: {sum(corrected_timesteps)}")
print(f"edit probability: {sum(corrected_timesteps) / len(corrected_timesteps)}")


In [ ]:
preds.shape

In [ ]:
context.shape

In [ ]:
plot_median = False

plt.figure(figsize=(6, 2))
ctx = context.squeeze().cpu().numpy() if hasattr(context, "cpu") else context.squeeze()
future_vals_timesteps = np.arange(context_length, context_length + prediction_length)

plt.plot(
    future_vals_timesteps,
    future_vals.squeeze(),
    "k--",
    linewidth=1,
    label="Ground Truth",
)
preds = preds.squeeze()
if preds.ndim == 1:
    preds = preds[None]
n, L = preds.shape
print(preds.shape)

plt.plot(np.arange(len(ctx)), ctx, "k", linewidth=1, label="Context")

if plot_median:
    plt.plot(
        np.arange(len(ctx), len(ctx) + L),
        np.median(preds, axis=0),
        color="tab:blue",
        linewidth=1,
        label="Median Prediction",
    )
else:
    for i in range(n):
        plt.plot(
            np.arange(len(ctx), len(ctx) + L),
            preds[i],
            color=DEFAULT_COLORS[i],
            linewidth=1,
        )

plt.xlabel("Timestep", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(plot_save_dir, f"predictions_{system_name}.pdf"))
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6, 3), sharex=True, height_ratios=[2, 1])

plot_median = True

timesteps = np.arange(len(corrected_timesteps))
print(timesteps.shape)
corrected = (
    corrected_timesteps.cpu().numpy()
    if hasattr(corrected_timesteps, "cpu")
    else corrected_timesteps
)

# --- First row: plot ground truth and predictions ---
ax1 = axes[0]
# Plot ground truth
ax1.plot(
    timesteps,
    future_vals.squeeze()[: len(timesteps)],
    "k--",
    linewidth=1,
    label="Ground Truth",
)
# Plot predictions
if plot_median:
    ax1.plot(
        timesteps,
        np.median(preds, axis=0)[: len(timesteps)],
        color="tab:blue",
        linewidth=1,
        label="Median Prediction",
    )
else:
    for i in range(n):
        ax1.plot(
            timesteps,
            preds[i][: len(timesteps)],
            color=DEFAULT_COLORS[i],
            linewidth=1,
        )
ax1.set_ylabel("x(t)", fontweight="bold")
ax1.legend(loc="best", fontsize=8)
ax1.set_title("Predictions", fontweight="bold")

# --- Second row: barcode plot for interventions ---
ax2 = axes[1]
for t, is_corr in zip(timesteps, corrected):
    if is_corr:
        ax2.axvline(t, color="black", linewidth=0.5, alpha=1.0)
ax2.set_xlim(timesteps[0] - 0.5, timesteps[-1] + 0.5)
ax2.set_ylim(0, 1)
ax2.set_yticks([])
ax2.set_xlabel("Timestep", fontweight="bold")
ax2.set_title("Interventions", fontweight="bold")

plt.tight_layout()
plt.show()
